# 🧹 Somali News Dataset — Cleaning Pipeline
**Thesis:** Automatic Somali Headline Generation Using Transformer-Based NLP Models

| Cell | Section |
|------|---------|
| 0    | Install dependencies |
| 1    | Imports & configuration |
| 2    | Load raw files |
| 3    | Raw data inspection |
| 4    | Cleaning functions |
| 5    | Clean headlines |
| 6    | Clean article bodies |
| 7    | Remove boilerplate & nav contamination |
| 8    | Handle missing values |
| 9    | Deduplication |
| 10   | Suspicious article detection |
| 11   | Category & source validation |
| 12   | Save cleaned files |
| 13   | Build model-ready dataset |
| 14   | Final summary report |

## Cell 0 — Install Dependencies

In [1]:
import subprocess, sys
for pkg in ['pandas','numpy','tqdm','tabulate']:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True)
    print('✔' if r.returncode==0 else '✘', pkg)
print('\nDone. Restart kernel if this was your first run.')

✔ pandas
✔ numpy
✔ tqdm
✔ tabulate

Done. Restart kernel if this was your first run.


## Cell 1 — Imports & Configuration

In [2]:
import pandas as pd
import numpy as np
import re, os, hashlib, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

# ─── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR     = Path('.')
RAW_DIR      = BASE_DIR / 'data' / 'raw'        # put your CSV files here
CLEANED_DIR  = BASE_DIR / 'data' / 'cleaned'
REVIEW_DIR   = BASE_DIR / 'data' / 'review'
REPORTS_DIR  = BASE_DIR / 'data' / 'reports'
for d in [RAW_DIR, CLEANED_DIR, REVIEW_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Categories ──────────────────────────────────────────────────────────────
VALID_CATEGORIES = ['siyaasad', 'amni', 'caalamka', 'ciyaaro']

# ─── Quality thresholds ──────────────────────────────────────────────────────
MIN_TITLE_CHARS   = 15    # titles shorter than this go to review
MIN_BODY_CHARS    = 150   # bodies shorter than this go to review
MIN_BODY_WORDS    = 30    # bodies with fewer words go to review
MAX_TITLE_CHARS   = 300   # unusually long titles go to review

# ─── Known nav/menu titles scraped accidentally from Goobjoog sidebar ────────
# Identified by analyzing the first rows of goobjoog__amni.csv
NAV_TITLES = {
    'Wararka Caalamka', 'Dhaqanka & Buugta', 'Aragtida Xorta Ah',
    'Arrimaha Ganacsiga', 'La-dagaallanka Al-Shabaab', 'Deegaanka & Cimilada',
    'Arrimaha Amniga', 'Dagaalka Dib-u-Xoreynta', 'Siyaasadda Xogdhawrka',
    'Wararka Maanta', 'Ciyaaraha', 'Caafimaad', 'Diinta',
}

# ─── Boilerplate phrases found in content (from analysis) ────────────────────
BOILERPLATE_PATTERNS = [
    r'nala soo xiriir.*',                          # contact form
    r'Your name.*Your email.*Subject',             # mustaqbal contact page
    r'waxa aad ka heli kartaa.*barnaamijka',       # promo text
    r'raac|subscribe|nagula soo xiriir|la soco.*facebook',  # social CTAs
    r'xuquuqda.*dhowrsantahay|all rights reserved', # copyright
    r'haddaad rabto.*war.*noogu soo dir',          # UGC prompts
    r'\bFacebook\b.*\bTwitter\b.*\bInstagram\b',  # share bar
    r'akhri.*wariye.*goobjoog',                    # site byline
    r'qore.*caasimada|caasimada.*qore',            # site byline
]

TODAY = datetime.now().strftime('%Y-%m-%d')
print('✅ Configuration loaded.')
print(f'   RAW_DIR     : {RAW_DIR.resolve()}')
print(f'   CLEANED_DIR : {CLEANED_DIR.resolve()}')
print(f'   Categories  : {VALID_CATEGORIES}')

✅ Configuration loaded.
   RAW_DIR     : E:\somali_cleaner\data\raw
   CLEANED_DIR : E:\somali_cleaner\data\cleaned
   Categories  : ['siyaasad', 'amni', 'caalamka', 'ciyaaro']


## Cell 2 — Load Raw Files

In [3]:
# ─── Scan RAW_DIR for CSV files ──────────────────────────────────────────────
# The notebook loads all CSV files found in data/raw/

REQUIRED_COLS = ['site', 'category', 'url', 'title', 'content']

raw_files = sorted(RAW_DIR.glob('*.csv'))
if not raw_files:
    print('⚠ No CSV files found in data/raw/')
else:
    frames = []
    for fp in raw_files:
        try:
            df = pd.read_csv(fp, dtype=str)
            missing = [c for c in REQUIRED_COLS if c not in df.columns]
            # Rename 'title' → 'headline' internally for clarity
            if 'title' in df.columns:
                df = df.rename(columns={'title': 'headline', 'content': 'body'})
            if missing:
                print(f'  ⚠ {fp.name}: missing columns {missing} — skipped')
                continue
            df['source_file'] = fp.name
            frames.append(df)
            print(f'  ✔ {fp.name:<45} {len(df):>6,} rows')
        except Exception as e:
            print(f'  ✘ {fp.name}: {e}')

    raw_df = pd.concat(frames, ignore_index=True)
    TOTAL_RAW = len(raw_df)
    print(f'\n  Total loaded: {TOTAL_RAW:,} rows across {len(frames)} files')

  ✔ bbc_somali__ciyaaro.csv                          245 rows
  ✔ caasimada__caalamka.csv                        4,049 rows
  ✔ caasimada__siyaasad.csv                        5,316 rows
  ✔ garoweonline__ciyaaro.csv                         11 rows
  ✔ goobjoog__amni.csv                             6,595 rows
  ✔ goobjoog__caalamka.csv                         1,519 rows
  ✔ goobjoog__ciyaaro.csv                              0 rows
  ✔ goobjoog__siyaasad.csv                         4,746 rows
  ✔ kooxda__ciyaaro.csv                            7,193 rows
  ✔ kooxdamanta__ciyaaro.csv                           8 rows
  ✔ kubadlive__ciyaaro.csv                           114 rows
  ✔ laacibnet__ciyaaro.csv                         1,094 rows
  ✔ mustaqbalmedia__caalamka.csv                   4,548 rows
  ✔ mustaqbalmedia__ciyaaro.csv                       92 rows
  ✔ puntlandpost__ciyaaro.csv                        226 rows
  ✔ sonna__ciyaaro.csv                                26 rows
  ✔ wara

## Cell 3 — Raw Data Inspection

In [4]:
# ─── Overview ────────────────────────────────────────────────────────────────
print('═'*65)
print('  RAW DATA INSPECTION REPORT')
print('═'*65)
print(f'\n  Total rows      : {len(raw_df):,}')
print(f'  Columns         : {list(raw_df.columns)}')

# Missing values
print('\n  ── Missing values ──────────────────────────────────────')
for col in ['headline','body','url','category','site']:
    if col in raw_df.columns:
        n = raw_df[col].isna().sum() + (raw_df[col].astype(str).str.strip() == '').sum()
        pct = n / len(raw_df) * 100
        flag = '⚠' if n > 0 else ' '
        print(f'  {flag} {col:<15}: {n:>5} missing ({pct:.1f}%)')

# Category distribution
print('\n  ── Category distribution ───────────────────────────────')
print(raw_df['category'].value_counts().to_string())

# Site distribution
print('\n  ── Site distribution ───────────────────────────────────')
print(raw_df['site'].value_counts().to_string())

# Headline length
raw_df['_hlen'] = raw_df['headline'].astype(str).str.len()
raw_df['_blen'] = raw_df['body'].astype(str).str.len()
raw_df['_bwords'] = raw_df['body'].astype(str).str.split().str.len()

print('\n  ── Headline length (chars) ─────────────────────────────')
print(raw_df['_hlen'].describe().round(1).to_string())
print(f'  < {MIN_TITLE_CHARS} chars : {(raw_df["_hlen"] < MIN_TITLE_CHARS).sum()} rows')
print(f'  > {MAX_TITLE_CHARS} chars : {(raw_df["_hlen"] > MAX_TITLE_CHARS).sum()} rows')

print('\n  ── Body length (words) ─────────────────────────────────')
print(raw_df['_bwords'].describe().round(1).to_string())
print(f'  < {MIN_BODY_WORDS} words  : {(raw_df["_bwords"] < MIN_BODY_WORDS).sum()} rows')

# Duplicates
dup_urls     = raw_df.duplicated(subset=['url']).sum()
dup_headline = raw_df.duplicated(subset=['headline']).sum()
print('\n  ── Duplicates ──────────────────────────────────────────')
print(f'  Duplicate URLs      : {dup_urls:,}')
print(f'  Duplicate headlines : {dup_headline:,}')
print('\n' + '═'*65)

═════════════════════════════════════════════════════════════════
  RAW DATA INSPECTION REPORT
═════════════════════════════════════════════════════════════════

  Total rows      : 35,797
  Columns         : ['id', 'site', 'category', 'url', 'headline', 'body', 'scraped_at', 'word_count', 'source_file']

  ── Missing values ──────────────────────────────────────
    headline       :     0 missing (0.0%)
    body           :     0 missing (0.0%)
    url            :     0 missing (0.0%)
    category       :     0 missing (0.0%)
    site           :     0 missing (0.0%)

  ── Category distribution ───────────────────────────────
category
caalamka    10116
siyaasad    10062
ciyaaro      9024
amni         6595

  ── Site distribution ───────────────────────────────────
site
goobjoog          12860
caasimada          9365
kooxda             7193
mustaqbalmedia     4640
laacibnet          1094
bbc_somali          245
puntlandpost        226
kubadlive           114
sonna                26
wa

## Cell 4 — Cleaning Functions

In [5]:
"""
All cleaning logic is defined here as reusable functions.
Run this cell once — all subsequent cells call these functions.
"""

# ─── Text normalisation ───────────────────────────────────────────────────────
def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces/newlines into a single space."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)   # newlines → space
    text = re.sub(r' {2,}', ' ', text)        # multi-space → single
    text = re.sub(r'[\u200b\u200c\u200d\ufeff\u00ad]', '', text)  # zero-width
    return text.strip()

def fix_punctuation(text: str) -> str:
    """Normalise punctuation spacing (no space before . , ; ?)."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r' +([.,:;!?])', r'\1', text)
    text = re.sub(r'([.,:;!?]) {2,}', r'\1 ', text)
    return text

def remove_ellipsis_tails(text: str) -> str:
    """
    Goobjoog excerpts end in '…' because the scraper caught the teaser.
    Remove trailing ellipsis + any partial sentence after it.
    """
    if not isinstance(text, str):
        return ''
    # Remove everything after last full stop if line ends with '…'
    text = re.sub(r'\s*…\s*$', '', text)
    text = re.sub(r'\s*\.\.\.\s*$', '', text)
    return text.strip()

# ─── Headline cleaning ────────────────────────────────────────────────────────
def clean_headline(text: str) -> str:
    """Full pipeline for a single headline string."""
    if not isinstance(text, str):
        return ''
    text = normalize_whitespace(text)
    text = fix_punctuation(text)
    # Remove 'Sawirro:' / 'Daawo:' / 'Muqaal:' prefixes (media labels)
    text = re.sub(r'^(Sawirro|Daawo|Muqaal|VIDEO|PHOTO|Akhriso|AKHRISO)\s*[:\-]?\s*',
                  '', text, flags=re.IGNORECASE)
    # Remove trailing site name suffixes  e.g. '– Caasimada Online'
    text = re.sub(r'\s*[|\-–—]\s*(Caasimada|Goobjoog|Mustaqbal|Radio Dalsan).*$',
                  '', text, flags=re.IGNORECASE)
    # Remove quote marks around the whole headline
    text = re.sub(r'^["\u201c\u201e](.+)["\u201d]$', r'\1', text)
    return normalize_whitespace(text)

# ─── Body cleaning ────────────────────────────────────────────────────────────
def remove_boilerplate(text: str) -> str:
    """
    Remove known boilerplate patterns detected during analysis.
    Patterns cover: social share bars, copyright lines,
    contact forms (mustaqbal), site bylines.
    """
    if not isinstance(text, str):
        return ''
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE | re.DOTALL)
    return text

def remove_social_mentions(text: str) -> str:
    """Remove Facebook/Twitter/Instagram share sentences."""
    if not isinstance(text, str):
        return ''
    # Remove sentences containing social media keywords
    sentences = re.split(r'(?<=[.!?]) +', text)
    social_kw = re.compile(
        r'\b(Facebook|Twitter|Instagram|YouTube|Telegram|WhatsApp|Threads)\b',
        re.IGNORECASE
    )
    clean_sents = [s for s in sentences if not social_kw.search(s)]
    return ' '.join(clean_sents)

def clean_body(text: str) -> str:
    """Full pipeline for a single article body."""
    if not isinstance(text, str):
        return ''
    text = normalize_whitespace(text)
    text = remove_boilerplate(text)
    text = remove_social_mentions(text)
    text = remove_ellipsis_tails(text)
    text = fix_punctuation(text)
    text = normalize_whitespace(text)  # second pass after removals
    return text

# ─── Nav contamination detector ───────────────────────────────────────────────
def is_nav_article(headline: str) -> bool:
    """
    Returns True if the headline matches a known navigation/menu item
    that was accidentally scraped (found in Goobjoog sidebar).
    """
    return str(headline).strip() in NAV_TITLES

def is_contact_page(body: str) -> bool:
    """Detect mustaqbal contact-form page scraped accidentally."""
    return bool(re.search(
        r'Your name.*Your email|nala soo xiriir.*email',
        str(body), re.IGNORECASE | re.DOTALL
    ))

# ─── Suspicious article checks ────────────────────────────────────────────────
def suspicious_flags(row) -> list:
    """
    Returns a list of issue strings for a row.
    Empty list = row is clean.
    """
    flags = []
    hl = str(row.get('headline_clean', ''))
    bd = str(row.get('body_clean', ''))

    if len(hl) < MIN_TITLE_CHARS:
        flags.append(f'short_headline({len(hl)})')
    if len(hl) > MAX_TITLE_CHARS:
        flags.append(f'long_headline({len(hl)})')
    if len(bd) < MIN_BODY_CHARS:
        flags.append(f'short_body({len(bd)})')
    bwords = len(bd.split())
    if bwords < MIN_BODY_WORDS:
        flags.append(f'few_words({bwords})')
    if not row.get('url') or str(row.get('url','')) == 'nan':
        flags.append('missing_url')
    if row.get('category','') not in VALID_CATEGORIES:
        flags.append(f'invalid_category({row.get("category","")})')
    # Repeated word check (>30% of words the same word)
    if bwords > 10:
        words = bd.lower().split()
        most_common_count = Counter(words).most_common(1)[0][1]
        if most_common_count / bwords > 0.30:
            flags.append('repetitive_content')
    return flags

# ─── Deduplication ────────────────────────────────────────────────────────────
def dedup_report(before: int, after: int, stage: str):
    removed = before - after
    print(f'  {stage:<35} removed {removed:>5,}  (before {before:,} → after {after:,})')

# ─── MD5 hash helper ─────────────────────────────────────────────────────────
def md5(text: str) -> str:
    return hashlib.md5(str(text).strip().lower().encode()).hexdigest()

print('✅ All cleaning functions defined.')

✅ All cleaning functions defined.


## Cell 5 — Clean Headlines

In [6]:
from tqdm.auto import tqdm
tqdm.pandas()

df = raw_df.copy()

print('Cleaning headlines...')
df['headline_clean'] = df['headline'].progress_apply(clean_headline)

# ── Before / after comparison sample ─────────────────────────────────────────
changed = df[df['headline'] != df['headline_clean']]
print(f'\n  Headlines modified: {len(changed):,}')
print('\n  Sample changes:')
for _, row in changed.sample(min(5, len(changed)), random_state=42).iterrows():
    print(f'  BEFORE: {str(row["headline"])[:90]}')
    print(f'  AFTER : {str(row["headline_clean"])[:90]}')
    print()

# ── Empty headline check ──────────────────────────────────────────────────────
empty_hl = (df['headline_clean'].str.strip() == '') | df['headline_clean'].isna()
print(f'  Empty after cleaning: {empty_hl.sum()}')

Cleaning headlines...


  0%|          | 0/35797 [00:00<?, ?it/s]


  Headlines modified: 1,360

  Sample changes:
  BEFORE: Video: Culumada degmada Warsheekh oo ka qeyliyey ‘fisqi uga yimaada Xamar’
  AFTER : Culumada degmada Warsheekh oo ka qeyliyey ‘fisqi uga yimaada Xamar’

  BEFORE: Akhriso: War-murtiyeedka Shirka Iskaashiga caalamiga ah ee Soomaaliya Oo Dhameystiran
  AFTER : War-murtiyeedka Shirka Iskaashiga caalamiga ah ee Soomaaliya Oo Dhameystiran

  BEFORE: “Wan Garaaci Karnaa England”
  AFTER : Wan Garaaci Karnaa England

  BEFORE: Sawirro: 60,000 oo ruux oo ka baracakay Hiiraan, kadib…
  AFTER : 60,000 oo ruux oo ka baracakay Hiiraan, kadib…

  BEFORE: Akhriso: Taariikhda Boqorkii Cuudka Axmed Xudeydi oo Ingiriiska Ku Geeriyooday
  AFTER : Taariikhda Boqorkii Cuudka Axmed Xudeydi oo Ingiriiska Ku Geeriyooday

  Empty after cleaning: 0


## Cell 6 — Clean Article Bodies

In [7]:
print('Cleaning bodies (this takes ~30 seconds)...')
df['body_clean'] = df['body'].progress_apply(clean_body)

# ── Stats after cleaning ──────────────────────────────────────────────────────
df['_bwords_clean'] = df['body_clean'].str.split().str.len()
df['_blen_clean']   = df['body_clean'].str.len()

print(f'\n  Body word counts after cleaning:')
print(df['_bwords_clean'].describe().round(1).to_string())

# ── Sample cleaned body ───────────────────────────────────────────────────────
print('\n  Sample (first article):')
sample = df[df['_bwords_clean'] > 50].iloc[0]
print(f'  HEADLINE : {sample["headline_clean"]}')
print(f'  BODY(100): {sample["body_clean"][:300]}')

Cleaning bodies (this takes ~30 seconds)...


  0%|          | 0/35797 [00:00<?, ?it/s]


  Body word counts after cleaning:
count    35797.0
mean       239.0
std        192.7
min          0.0
25%        129.0
50%        185.0
75%        278.0
max       2677.0

  Sample (first article):
  HEADLINE : Maxay dalalku ka faa'idaan marka ay u soo baxaan ka qeybgalka Koobka Adduunka?
  BODY(100): Xigashada Sawirka, Getty Images Dalalka si guul leh ugu soo baxa Koobka Adduunka ee FIFA waxay helaan sharaf weyn oo ay dunidu u hayso, sidoo kale faa'iidooyin dhaqaale iyo farxad weyn ayaa soo wajahda shacabkooda. Tani ma aha oo keliya guul ciyaareed, balse waa guul saameyn ku leh siyaasadda, dhaqa


## Cell 7 — Remove Boilerplate & Nav Contamination

In [8]:
n_before = len(df)
removed_rows = []

# ── 1. Flag nav/menu articles (Goobjoog sidebar contamination) ───────────────
nav_mask = df['headline_clean'].apply(is_nav_article)
nav_rows = df[nav_mask].copy()
nav_rows['removal_reason'] = 'nav_menu_contamination'
removed_rows.append(nav_rows)
df = df[~nav_mask]
print(f'  ✘ Nav/menu articles removed   : {len(nav_rows):,}')
print(f'    Examples: {list(nav_rows["headline_clean"].unique()[:5])}')

# ── 2. Flag contact-page articles (mustaqbalmedia false scrapes) ─────────────
contact_mask = df['body'].apply(is_contact_page)
contact_rows = df[contact_mask].copy()
contact_rows['removal_reason'] = 'contact_page_scrape'
removed_rows.append(contact_rows)
df = df[~contact_mask]
print(f'  ✘ Contact-page articles removed: {len(contact_rows):,}')

# ── 3. Remove rows where body is empty after cleaning ────────────────────────
empty_body_mask = (df['body_clean'].str.strip() == '') | df['body_clean'].isna()
empty_rows = df[empty_body_mask].copy()
empty_rows['removal_reason'] = 'empty_body_after_cleaning'
removed_rows.append(empty_rows)
df = df[~empty_body_mask]
print(f'  ✘ Empty body after clean       : {len(empty_rows):,}')

# ── 4. Remove rows where headline is empty after cleaning ────────────────────
empty_hl_mask = (df['headline_clean'].str.strip() == '') | df['headline_clean'].isna()
empty_hl_rows = df[empty_hl_mask].copy()
empty_hl_rows['removal_reason'] = 'empty_headline_after_cleaning'
removed_rows.append(empty_hl_rows)
df = df[~empty_hl_mask]
print(f'  ✘ Empty headline after clean   : {len(empty_hl_rows):,}')

dedup_report(n_before, len(df), 'Boilerplate/nav removal')

# Save removed rows for review
if removed_rows:
    removed_df = pd.concat(removed_rows, ignore_index=True)
    out = REVIEW_DIR / 'removed_boilerplate.csv'
    removed_df.to_csv(out, index=False)
    print(f'\n  Removed rows saved → {out}')

  ✘ Nav/menu articles removed   : 19
    Examples: ['Wararka Caalamka', 'Dhaqanka & Buugta', 'Aragtida Xorta Ah', 'Arrimaha Ganacsiga', 'La-dagaallanka Al-Shabaab']
  ✘ Contact-page articles removed: 8
  ✘ Empty body after clean       : 2
  ✘ Empty headline after clean   : 0
  Boilerplate/nav removal             removed    29  (before 35,797 → after 35,768)

  Removed rows saved → data\review\removed_boilerplate.csv


## Cell 8 — Handle Missing Values

In [9]:
n_before = len(df)

# Standardise NaN representations
for col in ['headline_clean', 'body_clean', 'url', 'category', 'site']:
    if col in df.columns:
        df[col] = df[col].replace(['nan', 'None', 'NaN', ''], np.nan)

# ── Report missing values ─────────────────────────────────────────────────────
print('  Missing value report (after cleaning):')
for col in ['headline_clean', 'body_clean', 'url', 'category', 'site']:
    if col in df.columns:
        n = df[col].isna().sum()
        print(f'  {col:<20}: {n:>5} missing')

# ── Drop rows with missing headline OR body ─────────────
critical_mask = df['headline_clean'].isna() | df['body_clean'].isna()
critical_rows = df[critical_mask].copy()
critical_rows['removal_reason'] = 'missing_headline_or_body'
critical_rows.to_csv(REVIEW_DIR / 'removed_missing.csv', index=False)
df = df[~critical_mask]

# ── Fill missing URL with placeholder ────────────────────────────────────────
df['url'] = df['url'].fillna('unknown')

dedup_report(n_before, len(df), 'Missing value removal')
print(f'\n  Rows with missing headline/body → {REVIEW_DIR}/removed_missing.csv')

  Missing value report (after cleaning):
  headline_clean      :     0 missing
  body_clean          :     0 missing
  url                 :     0 missing
  category            :     0 missing
  site                :     0 missing
  Missing value removal               removed     0  (before 35,768 → after 35,768)

  Rows with missing headline/body → data\review/removed_missing.csv


## Cell 9 — Deduplication

In [10]:
n_before = len(df)
all_dup_rows = []

# ── Stage 1: Exact duplicate URLs ────────────────────────────────────────────
dup_url_mask = df.duplicated(subset=['url'], keep='first') & (df['url'] != 'unknown')
all_dup_rows.append(df[dup_url_mask].assign(dup_reason='exact_url'))
df = df[~dup_url_mask]
dedup_report(n_before, len(df), 'Stage 1: Exact duplicate URLs')

# ── Stage 2: Exact duplicate cleaned headlines ────────────────────────────────
n2 = len(df)
df['_hl_hash'] = df['headline_clean'].str.strip().str.lower().apply(
    lambda x: md5(x) if isinstance(x, str) else ''
)
dup_hl_mask = df.duplicated(subset=['_hl_hash'], keep='first')
all_dup_rows.append(df[dup_hl_mask].assign(dup_reason='exact_headline'))
df = df[~dup_hl_mask]
dedup_report(n2, len(df), 'Stage 2: Exact duplicate headlines')

# ── Stage 3: Exact duplicate bodies ──────────────────────────────────────────
n3 = len(df)
df['_body_hash'] = df['body_clean'].apply(
    lambda x: md5(x[:500]) if isinstance(x, str) else ''
)  # hash first 500 chars — catches same article published twice
dup_body_mask = df.duplicated(subset=['_body_hash'], keep='first')
all_dup_rows.append(df[dup_body_mask].assign(dup_reason='duplicate_body'))
df = df[~dup_body_mask]
dedup_report(n3, len(df), 'Stage 3: Duplicate body (first 500 chars)')

# ── Save removed duplicates ───────────────────────────────────────────────────
if all_dup_rows:
    dup_df = pd.concat(all_dup_rows, ignore_index=True)
    dup_df.to_csv(REVIEW_DIR / 'removed_duplicates.csv', index=False)

total_removed = n_before - len(df)
print(f'\n  Total duplicates removed : {total_removed:,}')
print(f'  Remaining rows           : {len(df):,}')
print(f'  Saved to → {REVIEW_DIR}/removed_duplicates.csv')

  Stage 1: Exact duplicate URLs       removed   694  (before 35,768 → after 35,074)
  Stage 2: Exact duplicate headlines  removed    40  (before 35,074 → after 35,034)
  Stage 3: Duplicate body (first 500 chars) removed    42  (before 35,034 → after 34,992)

  Total duplicates removed : 776
  Remaining rows           : 34,992
  Saved to → data\review/removed_duplicates.csv


## Cell 10 — Suspicious Article Detection

In [11]:
print('Scanning for suspicious articles...')

df['_flags'] = df.apply(suspicious_flags, axis=1)
df['_flag_str'] = df['_flags'].apply(lambda f: '|'.join(f) if f else '')

suspicious_mask = df['_flag_str'] != ''
suspicious_df   = df[suspicious_mask].copy()
clean_df        = df[~suspicious_mask].copy()

print(f'  Suspicious articles : {len(suspicious_df):,}')
print(f'  Clean articles      : {len(clean_df):,}')

# ── Flag breakdown ────────────────────────────────────────────────────────────
print('\n  Flag breakdown:')
all_flags = [f for flags in df['_flags'] for f in flags]
flag_type = re.compile(r'^([a-z_]+)')
flag_counts = Counter(flag_type.match(f).group(1) for f in all_flags if flag_type.match(f))
for flag, count in flag_counts.most_common():
    print(f'  {flag:<30}: {count:,}')

# ── Save suspicious for manual review ────────────────────────────────────────
cols_to_save = ['site','category','url','headline_clean','body_clean','_flag_str','source_file']
suspicious_df[cols_to_save].to_csv(
    REVIEW_DIR / 'suspicious_articles.csv', index=False
)
print(f'\n  → Saved to {REVIEW_DIR}/suspicious_articles.csv')
print('  Review this file manually before deciding to include/exclude these rows.')

# ── Show samples ─────────────────────────────────────────────────────────────
if len(suspicious_df) > 0:
    print('\n  Sample suspicious rows:')
    for _, row in suspicious_df.head(4).iterrows():
        print(f'  FLAG : {row["_flag_str"]}')
        print(f'  TITLE: {str(row["headline_clean"])[:80]}')
        print(f'  BODY : {str(row["body_clean"])[:120]}')
        print()

Scanning for suspicious articles...
  Suspicious articles : 119
  Clean articles      : 34,873

  Flag breakdown:
  few_words                     : 103
  short_body                    : 43
  short_headline                : 16

  → Saved to data\review/suspicious_articles.csv
  Review this file manually before deciding to include/exclude these rows.

  Sample suspicious rows:
  FLAG : short_body(91)|few_words(20)
  TITLE: Take control of your cookies
  BODY : Have a browse to see what we use them for and how you can change your settings to suit you.

  FLAG : few_words(23)
  TITLE: Dhegeyso: Maxay Soomaaliya ka rajeyneysaa cayaaraha Cecafa U-15?
  BODY : Xulka 15-jirrada Soomaaliya ayaa berri la cayaari doona dhiggooda Tanzania. Haddaba waa sidee rejada Soomaaliya? waxaa f

  FLAG : few_words(22)
  TITLE: Rajada kooxaha daba ordaya Liverpool iyo wararka Premeir league
  BODY : Rajada kooxaha daba ordaya Liverpool iyo wararka Premeir league Waxaad ku dhageysataa barnaamijka Ciyaaraha ee 

## Cell 11 — Category & Source Validation

In [12]:
print('═'*60)
print('  CATEGORY & SOURCE VALIDATION')
print('═'*60)

# Work on the main clean_df
print('\n  Category distribution (clean dataset):')
cat_counts = clean_df['category'].value_counts()
for cat, n in cat_counts.items():
    bar = '█' * int(n / cat_counts.max() * 30)
    pct = n / len(clean_df) * 100
    print(f'  {cat:<15} {n:>6,}  {pct:>5.1f}%  {bar}')

# Invalid categories
invalid_cat = clean_df[~clean_df['category'].isin(VALID_CATEGORIES)]
if len(invalid_cat) > 0:
    print(f'\n  ⚠ Rows with unexpected category: {len(invalid_cat)}')
    print(invalid_cat['category'].value_counts().to_string())

print('\n  Source distribution (clean dataset):')
src_counts = clean_df['site'].value_counts()
for src, n in src_counts.items():
    bar = '█' * int(n / src_counts.max() * 30)
    pct = n / len(clean_df) * 100
    print(f'  {src:<20} {n:>6,}  {pct:>5.1f}%  {bar}')

print('\n  Category × Site breakdown:')
pivot = clean_df.groupby(['category','site']).size().unstack(fill_value=0)
print(pivot.to_string())

════════════════════════════════════════════════════════════
  CATEGORY & SOURCE VALIDATION
════════════════════════════════════════════════════════════

  Category distribution (clean dataset):
  caalamka        10,067   28.9%  ██████████████████████████████
  siyaasad         9,346   26.8%  ███████████████████████████
  ciyaaro          8,983   25.8%  ██████████████████████████
  amni             6,477   18.6%  ███████████████████

  Source distribution (clean dataset):
  goobjoog             12,615   36.2%  ██████████████████████████████
  caasimada             8,745   25.1%  ████████████████████
  kooxda                7,189   20.6%  █████████████████
  mustaqbalmedia        4,605   13.2%  ██████████
  laacibnet             1,091    3.1%  ██
  bbc_somali              240    0.7%  
  puntlandpost            223    0.6%  
  kubadlive               111    0.3%  
  sonna                    25    0.1%  
  wararka24                15    0.0%  
  kooxdamanta               8    0.0%  
  ga

## Cell 12 — Save Cleaned Files

In [14]:
# ── Final column selection ────────────────────────────────────────────────────
OUTPUT_COLS = ['site', 'category', 'url', 'headline_clean', 'body_clean', 'scraped_at']

final_cols = [c for c in OUTPUT_COLS if c in clean_df.columns]
save_df = clean_df[final_cols].copy()
save_df = save_df.rename(columns={'headline_clean': 'headline', 'body_clean': 'body'})

# ── Save per-category files ───────────────────────────────────────────────────
print('Saving per-category cleaned files...')
saved_counts = {}
for cat in save_df['category'].unique():
    cat_df = save_df[save_df['category'] == cat]
    out_path = CLEANED_DIR / f'{cat}_cleaned.csv'
    cat_df.to_csv(out_path, index=False)
    saved_counts[cat] = len(cat_df)
    print(f'  ✔ {cat:<15} → {out_path.name}  ({len(cat_df):,} rows)')

# ── Save suspicious separately ──────────
if len(suspicious_df) > 0:
    susp_save = suspicious_df[final_cols].copy().rename(
        columns={'headline_clean':'headline','body_clean':'body'}
    )
    susp_save.to_csv(REVIEW_DIR / 'suspicious_articles_full.csv', index=False)

print(f'\n  Total clean articles saved: {sum(saved_counts.values()):,}')
print(f'  Output folder: {CLEANED_DIR.resolve()}')

Saving per-category cleaned files...
  ✔ ciyaaro         → ciyaaro_cleaned.csv  (8,983 rows)
  ✔ caalamka        → caalamka_cleaned.csv  (10,067 rows)
  ✔ siyaasad        → siyaasad_cleaned.csv  (9,346 rows)
  ✔ amni            → amni_cleaned.csv  (6,477 rows)

  Total clean articles saved: 34,873
  Output folder: E:\somali_cleaner\data\cleaned


## Cell 13 — Build Model-Ready Dataset

In [ ]:
"""
Prepares the final NLP-ready format for headline generation fine-tuning.

  input_text  = '[CATEGORY] ' + first 512 words of body
  target_text = cleaned headline

The category prefix helps the model learn category-aware generation.
You can later add [SIYAASAD], [AMNI] etc. as special tokens.
"""

MAX_BODY_WORDS_FOR_INPUT = 512  # truncate to fit transformer context window

def build_input_text(row) -> str:
    cat_tag = f"[{str(row['category']).upper()}]"
    body_words = str(row['body']).split()
    truncated  = ' '.join(body_words[:MAX_BODY_WORDS_FOR_INPUT])
    return f"{cat_tag} {truncated}"

model_df = save_df.copy()
model_df['input_text']  = model_df.apply(build_input_text, axis=1)
model_df['target_text'] = model_df['headline']

# ── Save combined model-ready file ────────────────────────────────────────────
model_cols = ['site', 'category', 'url', 'input_text', 'target_text']
model_df[model_cols].to_csv(CLEANED_DIR / 'combined_model_ready.csv', index=False)

# ── Also save combined cleaned (all categories together) ─────────────────────
save_df.to_csv(CLEANED_DIR / 'combined_cleaned.csv', index=False)

print(f'  ✔ combined_model_ready.csv  ({len(model_df):,} rows)')
print(f'  ✔ combined_cleaned.csv      ({len(save_df):,} rows)')
print(f'\n  Sample model format:')
sample = model_df.iloc[5]
print(f'  INPUT  : {sample["input_text"][:200]}')
print(f'  TARGET : {sample["target_text"]}')
print()

## Cell 14 — Final Summary Report

In [15]:
import csv as csv_mod

print('═'*65)
print('  CLEANING PIPELINE — FINAL SUMMARY REPORT')
print('═'*65)

summary_rows = []

print(f'\n  {"Stage":<40} {"Count":>8}')
print(f'  {"-"*48}')

stages = [
    ('Raw articles loaded',             TOTAL_RAW),
    ('After boilerplate/nav removal',   None),
    ('After missing value removal',     None),
    ('After deduplication',             None),
    ('Suspicious (sent to review)',      len(suspicious_df)),
    ('Final clean articles',            len(save_df)),
]

for label, count in stages:
    val = f'{count:,}' if count is not None else '(see cells above)'
    print(f'  {label:<40} {val:>8}')
    summary_rows.append({'stage': label, 'count': count})

print(f'\n  {"Category":<20} {"Clean articles":>15} {"% of total":>12}')
print(f'  {"-"*48}')
for cat, n in save_df['category'].value_counts().items():
    pct = n / len(save_df) * 100
    print(f'  {cat:<20} {n:>15,} {pct:>11.1f}%')

print(f'\n  {"Source":<20} {"Clean articles":>15}')
print(f'  {"-"*36}')
for src, n in save_df['site'].value_counts().items():
    print(f'  {src:<20} {n:>15,}')

# ── Headline quality stats ────────────────────────────────────────────────────
hl_lens = save_df['headline'].str.len()
bd_words = save_df['body'].str.split().str.len()
print(f'\n  Headline length  — avg: {hl_lens.mean():.0f} chars  min: {hl_lens.min()}  max: {hl_lens.max()}')
print(f'  Body word count  — avg: {bd_words.mean():.0f} words  min: {bd_words.min()}  max: {bd_words.max()}')

# ── Save CSV report ───────────────────────────────────────────────────────────
report_path = REPORTS_DIR / f'cleaning_summary_{TODAY}.csv'
with open(report_path, 'w', newline='', encoding='utf-8') as f:
    w = csv_mod.DictWriter(f, fieldnames=['stage','count'])
    w.writeheader()
    w.writerows(summary_rows)

print(f'\n  Report saved → {report_path}')
print('\n  Output files:')
for fp in sorted(list(CLEANED_DIR.glob('*.csv')) + list(REVIEW_DIR.glob('*.csv')) + list(REPORTS_DIR.glob('*.csv'))):
    size_kb = fp.stat().st_size // 1024
    print(f'  {str(fp.relative_to(BASE_DIR)):<55} {size_kb:>6} KB')

print('\n' + '═'*65)
print('  ✅ Cleaning pipeline complete.')
print('═'*65)

═════════════════════════════════════════════════════════════════
  CLEANING PIPELINE — FINAL SUMMARY REPORT
═════════════════════════════════════════════════════════════════

  Stage                                       Count
  ------------------------------------------------
  Raw articles loaded                        35,797
  After boilerplate/nav removal            (see cells above)
  After missing value removal              (see cells above)
  After deduplication                      (see cells above)
  Suspicious (sent to review)                   119
  Final clean articles                       34,873

  Category              Clean articles   % of total
  ------------------------------------------------
  caalamka                      10,067        28.9%
  siyaasad                       9,346        26.8%
  ciyaaro                        8,983        25.8%
  amni                           6,477        18.6%

  Source                Clean articles
  ----------------------------